In [ ]:
fs = 48000;
SR = fs;
from IPython.display import Audio
from drumkit import *
import matplotlib.pyplot as plt
import numpy as np
import wave, struct

In [ ]:
## Fundamental functions
def float2int(fnumb):
    return int(np.floor(fnumb))

def fill_zeros(array,fill_length):
    return array + [0]*(fill_length - len(array))

def play(wave,fs=SR):
    return Audio(wave, rate=fs, normalize=False)

def noise(amp,length):
    return [amp*random() for z in range(length)]

def envelope(length,decay,A=1,B=0):
    assert 0 < decay <= 1, "DECAY MUST BE IN (0, 1]"
    return [ (A-B) * ( (decay*(1-z/length))/((1-2*decay)*z/length + decay) ) + B for z in range(length) ]

def normalize(sound):
    g = np.max(np.abs(sound)).tolist()
    return [ sound[z]/g for z in range(len(sound)) ]

def save_wav_stereo(path, wave_samples, fs):
    """Write a stereo 16-bit WAV. Refuses to clip instead of normalising."""
    Left, Right = wave_samples
    
    """ Checks max across both channels """
    peak = max(max(abs(z) for z in Left), max(abs(z) for z in Right))
    if peak > 1:
        raise ValueError(f"Peak {peak:.2f} exceeds 1: lower the gains.")
    
    """ Interleaves samples in pairs """
    interleaved = [sample for pair in zip(Left, Right) for sample in pair]
    
    with wave.open(str(path), "wb") as f:
        f.setnchannels(2) # 2 channels for stereo 
        f.setsampwidth(2)
        f.setframerate(fs)
        f.writeframes(struct.pack(
            "<" + "h" * len(interleaved),
            *(round(x * 32767) for x in interleaved),
        ))


In [ ]:
## Sound generators
def square_wave(f,fs,amp,length):

    T = 1/f[0];
    T_samples = float2int(T*fs);
    Thalf_samples = float2int(T_samples/2);

    wave = [];
    z = 0;

    while z < length:
        T = 1/max(f[z],50);
        T_samples = float2int(T*fs);
        Thalf_samples = float2int(T_samples/2);

        wave = wave + ( [-amp]* Thalf_samples + [amp]* Thalf_samples );
        z = z + T_samples;
        
    return fill_zeros(wave,length)

def make_cymb(length, env, att_fact):
    N = noise(1,length)
    att = envelope(length,att_fact,0,1)
    return [ (2*N[z]-1)*env[z]*att[z] for z in range(length) ]

def make_drum(length, freq, env, noise_amp):
    wave = square_wave(freq,fs,1,length)
    nose = noise(noise_amp,length)
    return [ (wave[z] + ( 2*nose[z]-1*noise_amp ) )/(1+noise_amp)*env[z] for z in range(length) ]

def make_pattern(rhythm,wave):
    silence = [0] * len(wave)
    sounds = [silence, wave]
    groove = []
    for z in rhythm:
        groove += sounds[z]
    return groove

def mix(sounds,gains,pans):
    assert len(sounds) == len(gains)
    return [ sum(sample*gain*pan for sample, gain, pan in zip(current_wave, gains, pans)) for current_wave in zip(*sounds) ];  

In [ ]:
## Main routine
def set_length(fs,tempo,bars,beats,subbeats):

    tempo_time = 60/tempo; # tempo in seconds
    beat_samples = float2int(fs*tempo_time); # tempo in samples
    L_subb = float2int(beat_samples/subbeats); # subbeat in samples
    L_bar = beat_samples * beats;
    
    return L_subb, L_bar

def drum_kit(fs, bars, L_bar, waves, rhythm, gains, pans):

    # Grooves
    sounds = []
    for i in range(len(rhythm)):
        pat = make_pattern(rhythm[i], waves[i])
        sounds.append(pat)

    # Mixdown
    Kit = mix(sounds,gains,pans)
    Kit = normalize(Kit)
    Kit = fill_zeros(Kit,L_bar) * bars

    # Result
    time = [ z/fs for z in range( len(Kit) ) ]
    plt.figure(figsize=(15,2))
    plt.plot(time,Kit)
    plt.xlim(0,time[-1])
    plt.ylim(-1.1, 1.1)
    plt.grid(True)
    
    return Kit

### The Ronettes - Be my baby

In [ ]:
# User parameters
tempo = float2int(129/2)+1;
bars = 2;
beats = 4;
subbeats = 8;

# Grooves
rhy_kick = [1,0,0,0,0,0,1,0, 1,0,0,0,0,0,0,0, 1,0,0,0,0,0,1,0, 1,0,0,0,0,0,0,0]
rhy_snre = [0,0,0,0,0,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,0,0,0,0, 0,0,1,0,1,0,1,0]
rhy_hhat = [1,0,1,0,1,0,1,0, 1,0,1,0,1,0,1,0, 1,0,1,0,1,0,1,0, 1,0,1,0,1,0,1,0]
rhy_ohat = [0,1,0,1,0,1,0,1, 0,1,0,1,0,1,0,1, 0,1,0,1,0,1,0,1, 0,1,0,1,0,1,0,1]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ohat];

# Gains
# g = [0.0, 0.0, 0.5, 0.2];
g = [0.4, 0.99, 0.99, 0.3];
p_L = [0.5, 0.5, 0.4, 0.4];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats, subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.1,500,25), envelope(L_subb,0.7), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.2,300,150), envelope(L_subb,0.8), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.01),0.080)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

OH = make_cymb(L_subb,envelope(L_subb,0.8,0.,0.12),0.001)
OH = fill_zeros(OH,L_subb)
OH = np.tanh(OH).tolist()

# Drum Kit
W = [Kick, Snare, HH, OH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_1 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_1,fs)

### Michelle Jackson - Billie Jean

In [ ]:
# User parameters
tempo = 117
bars = 2;
beats = 8;
subbeats = 4;

# Grooves
rhy_kick = [1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0]
rhy_snre = [0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
rhy_hhat = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
rhy_ohat = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ohat];

# Gains
# g = [0.0, 0.0, 0., 0.2];
g = [0.7, 0.99, 0.5, 0.7];
p_L = [0.5, 0.5, 0.8, 0.2];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.1,500,25), envelope(L_subb,0.2), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.2,300,150), envelope(L_subb,0.4), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.05),0.00002)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

OH1 = make_cymb(L_subb,envelope(L_subb,0.6,0.3,0.),0.4)
OH1 = fill_zeros(OH1,L_subb)


# Drum Kit
W = [Kick, Snare, HH, OH1]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_2 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_2,fs)

### Queen - Another One Bites the Dust

In [ ]:
# User parameters
tempo = 112
bars = 4;
beats = 4;
subbeats = 4;

# Grooves
rhy_kick = [1,0,0,0, 0,0,0,0, 1,0,0,0, 0,0,0,0]
rhy_snre = [0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
rhy_hhat = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
R = [rhy_kick, rhy_snre, rhy_hhat];

# Gains
g = [0.7, 0.99, 0.7];
p_L = [0.5, 0.5, 0.5];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.1,500,25), envelope(L_subb,0.2), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.15,240,80), envelope(L_subb,0.2), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.04),0.00002)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

# Drum Kit
W = [Kick, Snare, HH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_3 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_3,fs)

### Stevie Wonder - Superstition

In [ ]:
# User parameters
tempo = 100
bars = 2;
beats = 8;
subbeats = 8;

# Grooves
rhy_kick = [1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0]
rhy_snre = [0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0]
rhy_hhat = [1,0,0,0,1,0,0,0, 1,0,0,0,1,0,0,1, 1,0,0,1,1,0,0,0, 1,0,0,0,1,0,0,1, 1,0,0,0,1,0,0,0, 1,0,0,0,1,0,0,1, 1,0,0,0,1,0,0,1, 1,0,0,1,1,0,0,0]
R = [rhy_kick, rhy_snre, rhy_hhat,];

# Gains
g = [0.7, 0.99, 0.5];
p_L = [0.5, 0.3, 0.6];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.05,1020,35), envelope(L_subb,0.2), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.1,520,130), envelope(L_subb,0.75), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.25),0.05)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

# Drum Kit
W = [Kick, Snare, HH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_4 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_4,fs)

### Led Zeppelin - When the Levee Breaks (Intro)

In [ ]:
# User parameters
tempo = 72
bars = 4;
beats = 4;
subbeats = 4;

# Grooves
rhy_kick = [1,0,0,0, 0,0,0,1, 0,0,1,1, 0,0,0,0]
rhy_snre = [0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
rhy_hhat = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
rhy_ohat = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ohat];

# Gains
g = [0.7, 0.99, 0.1, 0.1];
p_L = [0.5, 0.3, 0.6, 0.];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.05,1020,35), envelope(L_subb,0.05), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.1,520,130), envelope(L_subb,0.05), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.2),0.01)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

OH = make_cymb(L_subb,envelope(L_subb,0.25),0.3)
OH = fill_zeros(OH,L_subb)
OH = np.tanh(OH).tolist()

# Drum Kit
W = [Kick, Snare, HH, OH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_5 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_5,fs)

### Led Zeppelin - When the Levee Breaks (Break)

In [ ]:
# User parameters
tempo = 72
bars = 1;
beats = 8;
subbeats = 4;

# Grooves
rhy_kick = [0,1,0,0, 0,1,0,0, 0,1,0,0, 0,1,0,0, 1,0,0,0, 0,0,0,1, 0,0,1,1, 0,0,0,0]
rhy_snre = [1,0,0,0, 1,0,0,0, 1,0,0,0, 1,0,1,1, 0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
rhy_hhat = [0,1,0,0, 0,1,0,0, 0,1,0,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
rhy_ohat = [0,0,0,0, 0,0,0,0, 0,0,0,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ohat];

# Gains
g = [0.7, 0.8, 0.1, 0.1];
p_L = [0.5, 0.3, 0.6, 0.];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.05,1020,35), envelope(L_subb,0.05), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.1,520,130), envelope(L_subb,0.05), 0)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.2),0.01)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

OH = make_cymb(L_subb,envelope(L_subb,0.25),0.3)
OH = fill_zeros(OH,L_subb)
OH = np.tanh(OH).tolist()

# Drum Kit
W = [Kick, Snare, HH, OH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_6 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_6,fs)

### Prince and the Revolution - When Doves Cry

In [ ]:
# User parameters
tempo = 64
bars = 2;
beats = 8;
subbeats = 4;

# Grooves
rhy_kick = [1,0,0,1, 0,0,0,0, 1,0,0,1, 0,0,0,0, 1,0,0,1, 0,0,0,0, 1,0,0,1, 0,0,0,0]
rhy_snre = [0,1,1,0, 0,0,0,0, 0,0,1,0, 0,0,0,0, 0,1,1,0, 0,0,0,0, 0,0,1,0, 0,0,0,0]
rhy_hhat = [0,0,1,0, 0,1,1,0, 0,0,1,0, 0,0,0,0, 0,0,1,0, 0,1,1,0, 0,0,1,0, 0,0,0,0]
rhy_ltom = [0,1,0,0, 1,0,0,0, 0,1,0,0, 1,1,1,0, 0,1,0,0, 1,0,0,0, 0,1,0,0, 1,1,1,0]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ltom];

# Gains
g = [0.5, 0.15, 0.3, 0.1];
p_L = [0.5, 0.5, 0.6, 0.5];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.05,1020,35), envelope(L_subb,0.2), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.25,450,440), envelope(L_subb,0.3), 1)
Snare = fill_zeros(Snare,L_subb)

HH = make_cymb(L_subb,envelope(L_subb,0.01),0.05)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

Tom = make_drum(L_subb, envelope(L_subb,0.05,280,35), envelope(L_subb,0.2), 0)
Tom = fill_zeros(Tom,L_subb)

# Drum Kit
W = [Kick, Snare, HH, Tom]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_7 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_7,fs)

### Toto - Rosanna

In [ ]:
# User parameters
tempo = 86;
bars = 4;
beats = 4;
subbeats = 8;

# Grooves                   |
rhy_kick = [1,0,0,0,0,0,0,1, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,0,0,0,0]
rhy_snre = [0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0]
rhy_hhat = [1,0,0,0,1,0,0,0, 1,0,0,0,1,0,0,0, 1,0,0,0,1,0,0,0, 1,0,0,0,1,0,0,0]
rhy_ohat = [0,0,0,1,0,0,0,1, 0,0,0,1,0,0,0,1, 0,0,0,1,0,0,0,1, 0,0,0,1,0,0,0,1]
R = [rhy_kick, rhy_snre, rhy_hhat, rhy_ohat];

# Gains
g = [0.99, 0.7, 0.3, 0.3];
p_L = [0.5, 0.5, 0.5, 0.5];
p_R = [1-p_L[0], 1-p_L[1], 1-p_L[2], 1-p_L[3]];

# Lengths
L_subb, L_bar = set_length(fs,tempo,bars,beats,subbeats);

# Drum One-shots
Kick = make_drum(L_subb, envelope(L_subb,0.05,1020,35), envelope(L_subb,0.2), 0)
Kick = fill_zeros(Kick,L_subb)
Kick = np.tanh(Kick).tolist()

Snare = make_drum(L_subb, envelope(L_subb,0.1,520,130), envelope(L_subb,0.5), 1)
Snare = fill_zeros(Snare,L_subb)
Snare = np.tanh(Snare).tolist()

OH = make_cymb(L_subb,envelope(L_subb,0.2),0.25)
OH = fill_zeros(OH,L_subb)
OH = np.tanh(OH).tolist()

HH = make_cymb(L_subb,envelope(L_subb,0.1),0.5)
HH = fill_zeros(HH,L_subb)
HH = np.tanh(HH).tolist()

# Drum Kit
W = [Kick, Snare, OH, HH]
Kit_L = drum_kit(fs, bars, L_bar, W, R, g, p_L); plt.title("Left channel");
Kit_R = drum_kit(fs, bars, L_bar, W, R, g, p_R); plt.title("Right channel");
Kit_stereo_8 = np.array([Kit_L,Kit_R])
# play(Kit_stereo_8,fs)

### Envelope formula derivation
- Linear fractional transformation function

    $$ y(x) = \frac{a \cdot x +b}{c \cdot x + d} $$
    
    Let $x = n/L$, where $n \in [0, L-1]$ is an array of integers. This is equivalent to t = n/fs.

- Boundary conditions

    To be a fade out function, it must met the following conditions
    - Condition 1: $y(0) = 1$
        $$ \frac{a \cdot (0/L)+b}{c \cdot (0/L)+d} = 1 \longrightarrow \frac{b}{d} = 1 \therefore b = d $$
    - Condition 2: $y(L) = 0$
        $$ \frac{a \cdot (L/L)+b}{c \cdot (L/L)+b} = 0 \longrightarrow a+b = 0 \therefore a = -b $$
    So our function can be rewriten as
        $$ y(n) = \frac{1 - n/L}{\alpha \cdot n/L + 1} \qquad \alpha = \frac{c}{b} $$

- Decay parameter $k$

    We add a decay parameter $k$ as $y(n) \to y(n,k)$ with the following constrains
    1) it affects the free parameter so that $\alpha \to \alpha(k)$,
    2) the function is fixed to k at halfpoint, i.e. $ y(L/2,k) = k $, and
    3) its values are $k \in [0,1]$ so that
        $$
        y(n,k) = \begin{cases}
        \text{Exponential} & 0.5 < k \leq 1 \\
        \text{Linear} & k = 0.5 \\
        \text{Logarithmic} & 0 \leq k < 0.5
        \end{cases}
        $$

    Setting the linear case
        $$ y(L/2,0.5) = 0.5 \longrightarrow 0.5 = \frac{1-0.5}{\alpha \cdot 0.5 + 1} \longrightarrow \frac{1}{\alpha + 2} \therefore \alpha = \frac{1-2 \cdot k}{k} $$
    
    yields the final fade-out function
        $$ y(n,k) = \frac{k \cdot (1 - n/L)}{(1-2 k) \cdot (n/L) + k} $$
    
    <!-- so that becomes
        $$
        y(n,k) = \begin{cases}
        \text{1} & k = 1 \\
        \text{Exponential} & 0.5 < k < 1 \\
        \text{(1 - n/L)} & k = 0.5 \\
        \text{Logarithmic} & 0 < k < 0.5 \\
        \text{0} & k = 0
        \end{cases}
        \qquad \forall n [0,L]
        $$ -->

- Start and End amplitudes $A$, $B$

    If a custom starting "A" and/or ending "B" points are desired, perform a simple interpolation, as
        $$ y(n,k,A,B) = (A - B)  \cdot y(n,k) + B $$

    therefore allowing fade-in/attack capabilities

In [ ]:
plt.figure(figsize=(15,3))
L = 4000

plt.subplot(1,3,1)
plt.plot(envelope(L,0.1))
plt.plot(envelope(L,0.5))
plt.plot(envelope(L,0.9))
plt.title(f"Fade-Out with A = 1 and B = 0");
plt.legend(["k = 0.1","k = 0.5","k = 0.9"]);
plt.ylim(0,1.1);
plt.grid(True);

plt.subplot(1,3,2)
A = 1; B = 0.5;
plt.plot(envelope(L,0.1,A,B))
plt.plot(envelope(L,0.5,A,B))
plt.plot(envelope(L,0.9,A,B))
plt.title(f"Fade-Out with A = {A} and B = {B}");
plt.legend(["k = 0.1","k = 0.5","k = 0.9"]);
plt.ylim(0,1.1);
plt.grid(True);

plt.subplot(1,3,3)
A = 0.0; B = 0.8;
plt.plot(envelope(L,0.1,A,B))
plt.plot(envelope(L,0.5,A,B))
plt.plot(envelope(L,0.9,A,B))
plt.title(f"Fade-In with A = {A} and B = {B}");
plt.legend(["k = 0.1","k = 0.5","k = 0.9"]);
plt.ylim(0,1.1);
plt.grid(True);

In [ ]:
All_kits = [Kit_stereo_1, Kit_stereo_2, Kit_stereo_3, Kit_stereo_4, Kit_stereo_5, Kit_stereo_6, Kit_stereo_7, Kit_stereo_8]
Full_kit_L = [sample for signal in All_kits for sample in signal[0]]
Full_kit_R = [sample for signal in All_kits for sample in signal[1]]

save_wav_stereo("solution.wav",[Full_kit_L, Full_kit_R],fs)

play(Full_kit,2*fs)